# Random Walk

**NIST/SEMATECH e-Handbook of Statistical Methods, Section 1.4.2.3**

Source: [https://www.itl.nist.gov/div898/handbook/eda/section4/eda4231.htm](https://www.itl.nist.gov/div898/handbook/eda/section4/eda4231.htm)

## Background

Cumulative sum of uniform random numbers minus 0.5.

### Analysis Goals

1. **Location:** What is a typical value?
2. **Variation:** How spread out are the data?
3. **Distribution:** What is the shape of the distribution?
4. **Randomness:** Are the data random (no autocorrelation or trend)?
5. **Outliers:** Are there any outliers in the data?

## Environment Setup

In [ ]:
# Check dependencies and install if missing
try:
    import numpy as np
    import scipy
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    !pip install numpy scipy pandas matplotlib seaborn
    import numpy as np
    import scipy
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

print(f'NumPy {np.__version__}, SciPy {scipy.__version__}, Pandas {pd.__version__}')
print(f'Matplotlib {plt.matplotlib.__version__}, Seaborn {sns.__version__}')

In [ ]:
# Quantum Explorer dark theme for matplotlib/seaborn
# Matches the site color scheme for visual consistency

import matplotlib.pyplot as plt
import seaborn as sns

QUANTUM_COLORS = {
    'background': '#0f1117',
    'surface': '#1a1d27',
    'accent': '#e06040',
    'teal': '#00a3a3',
    'text': '#e8e8f0',
    'text_secondary': '#9ca3af',
    'border': '#2a2d3a',
    'gradient_start': '#e06040',
    'gradient_end': '#00a3a3',
}

# Color cycle for multiple series
QUANTUM_PALETTE = ['#e06040', '#00a3a3', '#f0c040', '#a080e0', '#60c0a0', '#e080a0']

plt.rcParams.update({
    'figure.facecolor': QUANTUM_COLORS['background'],
    'axes.facecolor': QUANTUM_COLORS['surface'],
    'axes.edgecolor': QUANTUM_COLORS['border'],
    'axes.labelcolor': QUANTUM_COLORS['text'],
    'axes.titlecolor': QUANTUM_COLORS['text'],
    'xtick.color': QUANTUM_COLORS['text_secondary'],
    'ytick.color': QUANTUM_COLORS['text_secondary'],
    'text.color': QUANTUM_COLORS['text'],
    'grid.color': QUANTUM_COLORS['border'],
    'grid.alpha': 0.5,
    'figure.figsize': [10, 6],
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.prop_cycle': plt.cycler('color', QUANTUM_PALETTE),
})

sns.set_theme(style='darkgrid', rc={
    'axes.facecolor': QUANTUM_COLORS['surface'],
    'figure.facecolor': QUANTUM_COLORS['background'],
    'grid.color': QUANTUM_COLORS['border'],
    'text.color': QUANTUM_COLORS['text'],
    'axes.labelcolor': QUANTUM_COLORS['text'],
    'xtick.color': QUANTUM_COLORS['text_secondary'],
    'ytick.color': QUANTUM_COLORS['text_secondary'],
})

print('Quantum Explorer theme configured.')

In [ ]:
# Additional imports for statistical analysis
from scipy import stats
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

## Data Loading

Load the **RANDWALK.DAT** dataset from NIST (500 observations).

In [ ]:
# Load data from NIST .DAT file
DATA_FILE = 'RANDWALK.DAT'
GITHUB_URL = 'https://raw.githubusercontent.com/PatrykQuantumNomad/PatrykQuantumNomad/main/handbook/datasets/RANDWALK.DAT'

try:
    with open(DATA_FILE, 'r') as f:
        raw_text = f.read()
except FileNotFoundError:
    import urllib.request
    raw_text = urllib.request.urlopen(GITHUB_URL).read().decode()

# Skip header lines (25 lines)
header_lines = 25
data_text = '\n'.join(raw_text.strip().split('\n')[header_lines:])

df = pd.read_fwf(StringIO(data_text), header=None, names=['Y'])

print(f'Loaded {len(df)} rows')
assert len(df) == 500, f'Expected 500 rows, got {len(df)}'

In [ ]:
# Preview the first few rows
print(df.head(10))
print()
print(f'Shape: {df.shape}')
print(f'Data types:\n{df.dtypes}')

## Summary Statistics

Compute key descriptive statistics for the **Y** variable.

In [ ]:
# Summary statistics for Y
y = df['Y']

summary = pd.DataFrame({
    'Statistic': ['Mean', 'Std Dev', 'Median', 'Min', 'Max',
                  'Skewness', 'Kurtosis', 'N'],
    'Value': [
        y.mean(),
        y.std(ddof=1),
        y.median(),
        y.min(),
        y.max(),
        y.skew(),
        y.kurtosis(),
        len(y),
    ]
})

print(summary.to_string(index=False))

## 4-Plot Analysis

The 4-plot is a collection of four graphical EDA techniques whose purpose is to test 
the assumptions that underlie most measurement processes:

1. **Run Sequence Plot** (upper left) -- tests fixed location and variation
2. **Lag Plot** (upper right) -- tests randomness
3. **Histogram** (lower left) -- tests distributional assumptions
4. **Normal Probability Plot** (lower right) -- tests normality

In [ ]:
# 4-Plot: 4-Plot of Random Walk
y = df['Y'].values

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('4-Plot of Random Walk',
             fontsize=16, color=QUANTUM_COLORS['text'], y=0.98)

# 1. Run Sequence Plot (upper left)
ax = axes[0, 0]
ax.plot(range(len(y)), y, '.', color=QUANTUM_COLORS['accent'],
        markersize=3, alpha=0.7)
ax.axhline(y.mean(), color=QUANTUM_COLORS['teal'],
           linestyle='--', linewidth=1, label=f'Mean = {y.mean():.4f}')
ax.set_xlabel('Observation Number')
ax.set_ylabel('Y')
ax.set_title('Run Sequence Plot')
ax.legend(fontsize=9)

# 2. Lag Plot (upper right)
ax = axes[0, 1]
ax.scatter(y[:-1], y[1:], c=QUANTUM_COLORS['accent'],
           s=5, alpha=0.5)
ax.set_xlabel('Y(i)')
ax.set_ylabel('Y(i+1)')
ax.set_title('Lag Plot (lag=1)')

# 3. Histogram (lower left)
ax = axes[1, 0]
ax.hist(y, bins='auto', color=QUANTUM_COLORS['accent'],
        edgecolor=QUANTUM_COLORS['border'], alpha=0.8)
ax.set_xlabel('Y')
ax.set_ylabel('Frequency')
ax.set_title('Histogram')

# 4. Normal Probability Plot (lower right)
ax = axes[1, 1]
stats.probplot(y, dist='norm', plot=ax)
ax.get_lines()[0].set_color(QUANTUM_COLORS['accent'])
ax.get_lines()[1].set_color(QUANTUM_COLORS['teal'])
ax.set_title('Normal Probability Plot')

plt.tight_layout()
plt.show()

## Initial EDA Interpretation

The 4-plot reveals several important characteristics of this dataset:

- **Run Sequence Plot:** Shows a wandering, non-stationary pattern with no fixed location.
- **Lag Plot:** Shows a **strong linear relationship** between Y(i) and Y(i-1). 
This strong positive autocorrelation indicates the data are **NOT random** -- each value is highly 
dependent on the previous value.
- **Histogram:** Appears roughly symmetric but does not follow a bell curve.
- **Normal Probability Plot:** Shows some deviation from normality.

The key insight is the lag plot: the nearly perfect linear structure Y(i) vs Y(i-1) 
suggests an **AR(1) autoregressive model** is appropriate for this data.

## AR(1) Model Development

The lag plot reveals a strong linear relationship between Y(i) and Y(i-1), suggesting an **autoregressive model of order 1 (AR(1))**:

$$Y(i) = A_0 + A_1 \cdot Y(i-1) + E(i)$$

where:
- $A_0$ is the intercept
- $A_1$ is the autoregressive coefficient (slope)
- $E(i)$ is the random error term

We fit this model using **linear regression** of Y(i) on Y(i-1) via `scipy.stats.linregress`.

> **Note:** `linregress` returns `(slope, intercept, ...)`. In AR(1) notation: **slope = A1** (coefficient on Y(i-1)) and **intercept = A0**.

In [ ]:
# Fit AR(1) model: Y(i) = A0 + A1 * Y(i-1) + E(i)
# linregress(x, y) where x = Y(i-1) = y[:-1], y = Y(i) = y[1:]
from scipy.stats import linregress

y = df['Y'].values

# AR(1): regress Y(i) on Y(i-1)
slope, intercept, r_value, p_value, std_err = linregress(y[:-1], y[1:])

# slope = A1 (autoregressive coefficient)
# intercept = A0
A1 = slope
A0 = intercept
A1_se = std_err

# Compute t-value for A1
A1_t = A1 / A1_se

print(f'AR(1) Model: Y(i) = {A0:.6f} + {A1:.6f} * Y(i-1)')
print(f'A0 (intercept) = {A0:.6f}')
print(f'A1 (slope)     = {A1:.6f} +/- {A1_se:.6f}')
print(f'A1 t-value     = {A1_t:.3f}')
print(f'R-squared      = {r_value**2:.6f}')

In [ ]:
# Compute predicted values and residuals
y_pred = intercept + slope * y[:-1]
residuals = y[1:] - y_pred

# Residual standard deviation (ddof=2 for 2 estimated parameters: A0, A1)
resid_sd = np.std(residuals, ddof=2)
original_sd = np.std(y, ddof=1)

print(f'Number of residuals: {len(residuals)}')
print(f'Residual SD: {resid_sd:.4f}')
print(f'Original SD: {original_sd:.4f}')
print(f'Variability reduction: {original_sd/resid_sd:.1f}x')

### Parameter Comparison with NIST Reference Values

| Parameter | Computed | NIST Reference | Std Error |
|-----------|----------|---------------|-----------|
| A0 (intercept) | `computed above` | 0.050165 | +/- 0.024171 |
| A1 (slope) | `computed above` | 0.987087 | +/- 0.006313 |
| A1 t-value | `computed above` | 156.350 | - |
| Residual SD | `computed above` | 0.2931 | - |

The AR(1) model captures nearly all the autocorrelation structure in the random walk data. 
The autoregressive coefficient A1 = 0.987087 is very close to 1.0, indicating strong persistence.

## Predicted vs Original

Overlay the AR(1) predicted values on the original data to visually assess model fit.

In [ ]:
# Predicted vs Original overlay plot
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(range(len(y)), y, '.', color=QUANTUM_COLORS['accent'],
        markersize=3, alpha=0.5, label='Original Y')
ax.plot(range(1, len(y)), y_pred, '-', color=QUANTUM_COLORS['teal'],
        linewidth=0.8, alpha=0.8, label='AR(1) Predicted')

ax.set_xlabel('Observation Number')
ax.set_ylabel('Y')
ax.set_title('Random Walk: Original vs AR(1) Predicted Values')
ax.legend()
plt.tight_layout()
plt.show()

## Residual 4-Plot

Apply the standard 4-plot to the AR(1) residuals to validate that the model 
has adequately captured the autocorrelation structure. If the model is correct, 
the residuals should appear random.

In [ ]:
# Residual 4-Plot: run sequence, lag plot, histogram, normal probability plot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('4-Plot of AR(1) Residuals',
             fontsize=16, color=QUANTUM_COLORS['text'], y=0.98)

# 1. Run Sequence Plot of Residuals (upper left)
ax = axes[0, 0]
ax.plot(range(len(residuals)), residuals, '.', color=QUANTUM_COLORS['accent'],
        markersize=3, alpha=0.7)
ax.axhline(0, color=QUANTUM_COLORS['teal'],
           linestyle='--', linewidth=1, label='Zero')
ax.set_xlabel('Observation Number')
ax.set_ylabel('Residual')
ax.set_title('Run Sequence Plot of Residuals')
ax.legend(fontsize=9)

# 2. Lag Plot of Residuals (upper right)
ax = axes[0, 1]
ax.scatter(residuals[:-1], residuals[1:], c=QUANTUM_COLORS['accent'],
           s=5, alpha=0.5)
ax.set_xlabel('Residual(i)')
ax.set_ylabel('Residual(i+1)')
ax.set_title('Lag Plot of Residuals (lag=1)')

# 3. Histogram of Residuals (lower left)
ax = axes[1, 0]
ax.hist(residuals, bins='auto', color=QUANTUM_COLORS['accent'],
        edgecolor=QUANTUM_COLORS['border'], alpha=0.8)
ax.set_xlabel('Residual')
ax.set_ylabel('Frequency')
ax.set_title('Histogram of Residuals')

# 4. Normal Probability Plot of Residuals (lower right)
ax = axes[1, 1]
stats.probplot(residuals, dist='norm', plot=ax)
ax.get_lines()[0].set_color(QUANTUM_COLORS['accent'])
ax.get_lines()[1].set_color(QUANTUM_COLORS['teal'])
ax.set_title('Normal Probability Plot of Residuals')

plt.tight_layout()
plt.show()

## Residual Distribution Analysis

The NIST analysis shows that the AR(1) residuals follow a **uniform distribution**, 
not a normal distribution. This is because the original random walk was generated 
from cumulative sums of uniform random numbers minus 0.5.

We verify this by generating a **uniform probability plot** of the residuals.

In [ ]:
# Uniform probability plot of residuals
from scipy.stats import uniform

# Fit a uniform distribution to the residuals
loc, scale = uniform.fit(residuals)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Uniform probability plot
ax = axes[0]
res = stats.probplot(residuals, dist='uniform', plot=ax)
ax.get_lines()[0].set_color(QUANTUM_COLORS['accent'])
ax.get_lines()[1].set_color(QUANTUM_COLORS['teal'])
ax.set_title('Uniform Probability Plot of Residuals')

# Right: Compare with normal probability plot
ax = axes[1]
res_n = stats.probplot(residuals, dist='norm', plot=ax)
ax.get_lines()[0].set_color(QUANTUM_COLORS['accent'])
ax.get_lines()[1].set_color(QUANTUM_COLORS['teal'])
ax.set_title('Normal Probability Plot of Residuals (for comparison)')

plt.tight_layout()
plt.show()

print(f'Uniform fit: loc={loc:.4f}, scale={scale:.4f}')
print('The uniform probability plot should show points close to the diagonal,')
print('confirming the residuals follow a uniform distribution.')

## Conclusions

### Key Findings

1. **The data are NOT random.** The lag plot shows strong linear autocorrelation 
between successive values Y(i) and Y(i-1).

2. **AR(1) model captures the autocorrelation.** The model Y(i) = A0 + A1*Y(i-1) + E(i) 
with A0 = 0.050165 and A1 = 0.987087 (t = 156.350) explains nearly all the serial dependence.

3. **Variability reduced 7x.** The residual standard deviation (0.2931) is approximately 
7 times smaller than the original standard deviation (2.079), demonstrating the model 
captures most of the data structure.

4. **Residuals are uniform, not normal.** The residual distribution follows a uniform pattern, 
consistent with the data-generating process (cumulative sum of uniform random numbers).

5. **Residual 4-plot confirms model adequacy.** The residuals show no remaining autocorrelation, 
fixed location near zero, and constant variation.

**Reference:** [NIST/SEMATECH e-Handbook, Section 1.4.2.3](https://www.itl.nist.gov/div898/handbook/eda/section4/eda4231.htm)